<a href="https://colab.research.google.com/github/Guympele/Data-Science-Portfolio/blob/main/Document_de_Synth%C3%A8se_de_Assurances.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Automatisation du Rapprochement Financier des Assurances

Ce programme est conçu pour automatiser le rapprochement financier des assurances en consolidant les données de facturation et de paiement issues de fichiers Excel 2025 et 2026. Il nettoie les données, classifie les factures, calcule des synthèses, effectue des contrôles rigoureux et génère un rapport Excel professionnel avec trois feuilles détaillées : 'Synthèse', 'Réclamations' et 'Données complètes'.

In [18]:
# Install necessary libraries
!pip install pandas openpyxl numpy

### Étape 1: Importation des Fichiers Excel

Veuillez importer les deux fichiers Excel de l'assurance pour les années 2025 et 2026. Le programme tentera de déterminer l'année et le nom de l'assurance à partir des noms de fichiers.

In [19]:
import pandas as pd
import numpy as np
from google.colab import files
import re
import io
import unicodedata # Import unicodedata for accent removal

# Global variable to store the insurance name, to be identified from files
INSURANCE_NAME = None

def nettoyer_noms_colonnes(df):
    """Nettoie les noms de colonnes d'un DataFrame en supprimant les espaces inutiles,
    en les passant en minuscule, en remplaçant les espaces par des underscores, et en supprimant les accents.
    """
    cleaned_columns = {}
    for col in df.columns:
        # Convert to string to handle non-string column names (e.g., numbers)
        col_str = str(col)
        # Remove leading/trailing spaces
        cleaned_col = col_str.strip()
        # Convert to lowercase
        cleaned_col = cleaned_col.lower()
        # Remove accents
        cleaned_col = unicodedata.normalize('NFKD', cleaned_col).encode('ascii', 'ignore').decode('utf-8')
        # Replace spaces with underscores and remove multiple spaces
        cleaned_col = re.sub(r'\s+', ' ', cleaned_col).replace(' ', '_')
        cleaned_columns[col] = cleaned_col
    df = df.rename(columns=cleaned_columns)
    return df

def charger_fichiers():
    """Charge les fichiers Excel 2025 et 2026 depuis l'utilisateur,
    nettoie les noms de colonnes et identifie l'année et le nom de l'assurance.
    """
    print("Veuillez importer les deux fichiers Excel de l'assurance (2025 et 2026):")
    uploaded = files.upload()

    df_2025_local = None
    df_2026_local = None
    global INSURANCE_NAME

    file_names = list(uploaded.keys())

    if len(file_names) != 2:
        raise ValueError("Deux fichiers Excel (2025 et 2026) doivent être téléchargés.")

    print("\nFichiers détectés:")

    for fname in file_names:
        print(f"- {fname}")
        try:
            # Read a small portion of the file to determine the header row dynamically
            temp_df = pd.read_excel(io.BytesIO(uploaded[fname]), header=None, nrows=10)
            header_row_index = 0 # Default to row 0

            # Define a more comprehensive list of potential header keywords (case-insensitive)
            # These keywords should be unaccented to match the cleaned column names
            potential_header_keywords = [
                'id', 'facture', 'montant_assurance', 'montant_paye', 'decompte_vu',
                'identifiant', 'numero_facture', 'num_facture', 'invoice',
                'part_assurance', 'montant_pris_en_charge', 'prise_en_charge',
                'montant_regle', 'paiement', 'reglement', 'montant_reglement',
                'decompte', 'reglement_disponible', 'paiement_vu',
                'patient', 'mois', 'date_facture', 'assurance', 'montant_brut', 'ticket', 'moderateur', 'solde'
            ]

            max_found_keywords = 0
            best_header_row_index = 0

            for i in range(temp_df.shape[0]):
                # Convert row to string, handle potential NaN, and make lowercase for robust search
                current_row_values = temp_df.iloc[i].astype(str).apply(lambda x: unicodedata.normalize('NFKD', x).encode('ascii', 'ignore').decode('utf-8').lower()).tolist()
                found_keywords_count = 0
                for keyword in potential_header_keywords:
                    if any(keyword in str(val) for val in current_row_values):
                        found_keywords_count += 1

                if found_keywords_count > max_found_keywords:
                    max_found_keywords = found_keywords_count
                    best_header_row_index = i

            header_row_index = best_header_row_index
            print(f"Header for {fname} detected at row (0-indexed): {header_row_index}")

            # Re-read the full Excel file using the identified header row
            df = pd.read_excel(io.BytesIO(uploaded[fname]), header=header_row_index)
            df = nettoyer_noms_colonnes(df)

            # Try to infer year from filename
            year = None
            year_match = re.search(r'20(2[5-6])', fname)
            if year_match:
                year = int(year_match.group(0))

            insurance_match = re.search(r'([a-zA-Z]+)_?20(2[5-6])', fname) # More robust regex
            if insurance_match and not INSURANCE_NAME:
                INSURANCE_NAME = insurance_match.group(1).capitalize()

            if year == 2025:
                df_2025_local = df
                print(f"Fichier 2025 assigné : {fname}")
            elif year == 2026:
                df_2026_local = df
                print(f"Fichier 2026 assigné : {fname}")
            else:
                user_year = input(f"Impossible de déterminer l'année pour '{fname}'. Quel fichier correspond à 2025 ou 2026? Entrez 2025 ou 2026: ")
                if user_year == '2025':
                    df_2025_local = df
                elif user_year == '2026':
                    df_2026_local = df
                else:
                    raise ValueError(f"Année invalide fournie pour {fname}. Le processus ne peut pas continuer.")

        except Exception as e:
            print(f"Erreur lors du traitement du fichier {fname}: {e}")
            raise

    if df_2025_local is None or df_2026_local is None:
        raise ValueError("Assurez-vous d'avoir téléchargé un fichier pour 2025 et un pour 2026.")

    if not INSURANCE_NAME:
        print("Impossible de déterminer le nom de l'assurance à partir des noms de fichiers. ")
        INSURANCE_NAME = input("Veuillez entrer le nom de l'assurance (ex: Activa, Sanlam): ").strip().capitalize()
        if not INSURANCE_NAME:
            raise ValueError("Le nom de l'assurance est obligatoire pour la génération du rapport.")

    print(f"\nNom de l'assurance identifié: {INSURANCE_NAME}")

    return df_2025_local, df_2026_local

# The global assignments for df_2025 and df_2026 are necessary here because these
# variables are used outside this function scope, specifically in cell 059690c4.
# This is typically bad practice, but given the Colab notebook cell execution model,
# and to avoid changing the signature of subsequent functions, this is the most practical way.
df_2025, df_2026 = charger_fichiers()

print("\nChargement et nettoyage initial des colonnes terminés.")
print("Aperçu des données 2025 après nettoyage des colonnes:")
display(df_2025.head())
print("\nAperçu des données 2026 après nettoyage des colonnes:")
display(df_2026.head())

Veuillez importer les deux fichiers Excel de l'assurance (2025 et 2026):


Saving prudential_Bénéficial_2025.xlsx to prudential_Bénéficial_2025.xlsx
Saving Prudential_Bénéficial_2026.xlsx to Prudential_Bénéficial_2026.xlsx

Fichiers détectés:
- prudential_Bénéficial_2025.xlsx
Header for prudential_Bénéficial_2025.xlsx detected at row (0-indexed): 0
Fichier 2025 assigné : prudential_Bénéficial_2025.xlsx
- Prudential_Bénéficial_2026.xlsx
Header for Prudential_Bénéficial_2026.xlsx detected at row (0-indexed): 0
Fichier 2026 assigné : Prudential_Bénéficial_2026.xlsx

Nom de l'assurance identifié: Ficial

Chargement et nettoyage initial des colonnes terminés.
Aperçu des données 2025 après nettoyage des colonnes:


,id,mois,facture,date_facture,patient,assurance,montant_brut,ticket,moderateur,montant_assurance,decompte_vu,montant_paye,solde
0,ASS-2025-08-000001,Aout,912559.0,22/août/25,TAKAM Magoum emelie josiane,PRUDENTIAL BENEFICIAL,15300.0,80.0,3060.0,12240.0,Non,0.0,12240.0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,TOTAL,15300.0,NaN,3060.0,12240.0,NaN,0.0,12240.0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ASS-2025-09-000001,Septembre,929019.0,30/sept./25,OWONA Andela,PRUDENTIAL BENEFICIAL,228400.0,100.0,0.0,228400.0,Non,0.0,228400.0



Aperçu des données 2026 après nettoyage des colonnes:


,id,mois,facture,date_facture,patient,assurance,montant_brut,ticket,moderateur,montant_assurance,decompte_vu,montant_paye,solde
0,ASS-2026-01-000275,Janvier,976390.0,2026-01-15 00:00:00,MANGWA Sime chantal,PRUDENTIAL BENEFICIAL,53400.0,80.0,10680.0,42720.0,Oui,42720.0,0.0
1,ASS-2026-01-000276,Janvier,980289.0,2026-01-24 00:00:00,NGO Toga marie,PRUDENTIAL BENEFICIAL,6400.0,80.0,1280.0,5120.0,Non,0.0,5120.0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,TOTAL,59800.0,NaN,11960.0,47840.0,NaN,42720.0,5120.0
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
# Nettoyer le nom de l'assurance si l'utilisateur a entré une description
if 'synthèse' in INSURANCE_NAME.lower() or 'assurance' in INSURANCE_NAME.lower():
    INSURANCE_NAME = INSURANCE_NAME.lower().replace("synthèse de l'assurance ", "").capitalize()
    print(f"Nom de l'assurance corrigé: {INSURANCE_NAME}")


In [21]:
def identifier_colonnes(df):
    """Identifie et standardise les noms de colonnes clés en gérant les variations.
    Retourne un dictionnaire de mappage des colonnes identifiées.
    """
    standard_cols = {
        'id': ['id', 'identifiant', 'numero_facture', 'num_facture'],
        'mois': ['mois', 'month'],
        'facture': ['facture', 'invoice'],
        'date_facture': ['date_facture', 'date_invoice', 'datefacture'],
        'patient': ['patient', 'beneficiaire'],
        'assurance': ['assurance', 'compagnie'],
        'montant_brut': ['montant_brut', 'brut'],
        'ticket': ['ticket', 'ticket_moderateur', 'ticket_mod'],
        'moderateur': ['moderateur', 'taux_moderateur'],
        'montant_assurance': ['montant_assurance', 'part_assurance', 'montant_pris_en_charge', 'prise_en_charge'],
        'decompte_vu': ['decompte_vu', 'decompte', 'reglement_disponible', 'paiement_vu'],
        'montant_paye': ['montant_paye', 'montant_regle', 'paiement', 'reglement', 'montant_reglement'],
        'solde': ['solde', 'ecart_source']
    }

    identified_cols = {}
    # Use the exact column names from the DataFrame, which should already be cleaned by nettoyer_noms_colonnes
    df_cols = list(df.columns)

    print(f"DEBUG (identifier_colonnes): Colonnes du DataFrame: {df_cols}")

    for std_name, possible_names in standard_cols.items():
        found = False
        for p_name in possible_names:
            if p_name in df_cols:
                original_cleaned_name = p_name # Since df_cols already contain cleaned names
                identified_cols[std_name] = original_cleaned_name
                found = True
                print(f"DEBUG (identifier_colonnes): Trouvé '{std_name}' comme '{p_name}'")
                break
        # If not found, check if the standard name itself is present as a column
        if not found and std_name in df_cols:
            original_cleaned_name = std_name
            identified_cols[std_name] = original_cleaned_name
            print(f"DEBUG (identifier_colonnes): Trouvé '{std_name}' directement")

    print(f"DEBUG (identifier_colonnes): Colonnes identifiées: {identified_cols}")

    # Check for mandatory columns and raise error if not found
    mandatory_cols = ['id', 'montant_assurance', 'montant_paye']
    for col in mandatory_cols:
        if col not in identified_cols:
            raise ValueError(f"Colonne obligatoire introuvable: '{col}' (peut correspondre à {standard_cols.get(col, col)}). Veuillez vérifier vos fichiers source.")

    return identified_cols

def ajouter_annee(df, year):
    """Ajoute une colonne 'annee' au DataFrame.
    """
    df['annee'] = year
    return df

def exclure_lignes_total(df, identified_cols):
    """Exclut les lignes qui sont des totaux ou n'ont pas d'ID de facture.
    """
    # Use the identified 'id' column name
    id_col = identified_cols.get('id')

    # If 'id' column is identified, filter out rows where 'id' is empty
    if id_col and id_col in df.columns:
        df_filtered = df[df[id_col].notna() & (df[id_col] != 'TOTAL') & (df[id_col] != 'Total')]
    else:
        # Fallback if 'id' column not clearly identified or present, try to filter by 'facture' and 'montant_brut'
        print("Avertissement: La colonne 'id' n'a pas pu être clairement identifiée. Tentative de filtrage basée sur 'facture' et 'montant_brut'.")
        facture_col = identified_cols.get('facture', 'facture') # Use a common fallback
        montant_brut_col = identified_cols.get('montant_brut', 'montant_brut') # Use a common fallback

        df_filtered = df[df[facture_col].notna() & (df[facture_col].astype(str).str.lower() != 'total')]
        # Also filter out rows where 'montant_brut' is NaN or 0, assuming total rows might have these characteristics
        df_filtered = df_filtered[df_filtered[montant_brut_col].notna() & (df_filtered[montant_brut_col] != 0)]

    return df_filtered

def parse_multi_format_date(date_value):
    """Parse les dates en gérant plusieurs formats, y compris les abréviations de mois en français.
    """
    if pd.isna(date_value):
        return pd.NaT
    if isinstance(date_value, pd.Timestamp): # Already a datetime object
        return date_value

    date_str = str(date_value).strip().lower()

    # Case 1: YYYY-MM-DD format (like 2026-01-15)
    try:
        return pd.to_datetime(date_str, format='%Y-%m-%d')
    except ValueError:
        pass # Try next format

    # Case 2: DD/Mon./YY format with French month abbreviations (like 10/janv./25)
    french_month_map = {
        'janv.': 'jan', 'févr.': 'feb', 'mars': 'mar', 'avr.': 'apr',
        'mai': 'may', 'juin': 'jun', 'juil.': 'jul', 'août': 'aug',
        'sept.': 'sep', 'oct.': 'oct', 'nov.': 'nov', 'déc.': 'dec'
    }
    # Replace French abbreviations with English ones
    for fr_abbr, en_abbr in french_month_map.items():
        if fr_abbr in date_str:
            date_str = date_str.replace(fr_abbr, en_abbr)
            break # Assume only one month abbreviation per string

    try:
        # Format DD/Mon/YY (e.g., 10/jan/25)
        return pd.to_datetime(date_str, format='%d/%b/%y')
    except ValueError:
        pass # Try general parser as a last resort

    # Case 3: General parsing (e.g., if other formats appear, or for safety)
    return pd.to_datetime(date_str, errors='coerce')

def nettoyer_donnees(df, identified_cols):
    """Convertit les colonnes en types numériques, gère les dates et les valeurs manquantes.
    """
    df_cleaned = df.copy()

    # Convert monetary columns to numeric, coercing errors to NaN
    for col_std, col_actual in identified_cols.items():
        if col_std in ['montant_brut', 'montant_assurance', 'montant_paye', 'solde', 'ticket', 'moderateur']:
            df_cleaned[col_actual] = pd.to_numeric(df_cleaned[col_actual], errors='coerce').fillna(0)

    # Convert date column to datetime using the robust parser
    date_facture_col = identified_cols.get('date_facture')
    if date_facture_col and date_facture_col in df_cleaned.columns:
        df_cleaned[date_facture_col] = df_cleaned[date_facture_col].apply(parse_multi_format_date)

    # Ensure 'decompte_vu' is treated as boolean or string and fillna
    decompte_vu_col = identified_cols.get('decompte_vu')
    if decompte_vu_col and decompte_vu_col in df_cleaned.columns:
        df_cleaned[decompte_vu_col] = df_cleaned[decompte_vu_col].astype(str).replace({'nan': 'Non renseigné', 'NaT': 'Non renseigné', '': 'Non renseigné'}).fillna('Non renseigné')

    # Fill other relevant string columns with 'Non renseigné'
    for col_std, col_actual in identified_cols.items():
        if col_std in ['id', 'facture', 'patient', 'assurance', 'mois']:
            if col_actual in df_cleaned.columns:
                df_cleaned[col_actual] = df_cleaned[col_actual].astype(str).fillna('Non renseigné')

    return df_cleaned

# --- Main processing steps after file loading ---

# 5. Combiner les deux années
df_2025 = ajouter_annee(df_2025, 2025)
df_2026 = ajouter_annee(df_2026, 2026)

df_combined = pd.concat([df_2025, df_2026], ignore_index=True)

print("\nFichiers 2025 et 2026 combinés. Identification des colonnes en cours...")

# Identify columns using the combined DataFrame to ensure consistent mapping
# Pass df_combined to ensure all potential column names across both years are considered
identified_cols = identifier_colonnes(df_combined)

# Map the combined DataFrame columns to the identified standard names
# This ensures that all subsequent operations use consistent column names
df_combined = df_combined.rename(columns={v: k for k, v in identified_cols.items()})

print("Colonnes clés identifiées et renommées.")
print(f"Mappage des colonnes : {identified_cols}")

# Exclure les lignes 'TOTAL'
df_combined = exclure_lignes_total(df_combined, identified_cols) # Pass the original identified_cols for safety if renaming was an issue

print("Lignes 'TOTAL' exclues.")

# Nettoyer les données (conversion de type, gestion des NaN)
# Now that columns are renamed, pass the df_combined with new names to nettoyer_donnees
df_combined = nettoyer_donnees(df_combined, {k:k for k in identified_cols.keys()}) # Pass a simplified map since columns are already renamed

print("Nettoyage des données terminé.")

print("\nAperçu des données combinées et nettoyées:")
display(df_combined.head())

print(f"Nombre total de factures après nettoyage: {len(df_combined)}")


Fichiers 2025 et 2026 combinés. Identification des colonnes en cours...
DEBUG (identifier_colonnes): Colonnes du DataFrame: ['id', 'mois', 'facture', 'date_facture', 'patient', 'assurance', 'montant_brut', 'ticket', 'moderateur', 'montant_assurance', 'decompte_vu', 'montant_paye', 'solde', 'annee']
DEBUG (identifier_colonnes): Trouvé 'id' comme 'id'
DEBUG (identifier_colonnes): Trouvé 'mois' comme 'mois'
DEBUG (identifier_colonnes): Trouvé 'facture' comme 'facture'
DEBUG (identifier_colonnes): Trouvé 'date_facture' comme 'date_facture'
DEBUG (identifier_colonnes): Trouvé 'patient' comme 'patient'
DEBUG (identifier_colonnes): Trouvé 'assurance' comme 'assurance'
DEBUG (identifier_colonnes): Trouvé 'montant_brut' comme 'montant_brut'
DEBUG (identifier_colonnes): Trouvé 'ticket' comme 'ticket'
DEBUG (identifier_colonnes): Trouvé 'moderateur' comme 'moderateur'
DEBUG (identifier_colonnes): Trouvé 'montant_assurance' comme 'montant_assurance'
DEBUG (identifier_colonnes): Trouvé 'decompte_v

,id,mois,facture,date_facture,patient,assurance,montant_brut,ticket,moderateur,montant_assurance,decompte_vu,montant_paye,solde,annee
0,ASS-2025-08-000001,Aout,912559.0,2025-08-22,TAKAM Magoum emelie josiane,PRUDENTIAL BENEFICIAL,15300.0,80.0,3060.0,12240.0,Non,0.0,12240.0,2025
4,ASS-2025-09-000001,Septembre,929019.0,2025-09-30,OWONA Andela,PRUDENTIAL BENEFICIAL,228400.0,100.0,0.0,228400.0,Non,0.0,228400.0,2025
5,ASS-2025-09-000002,Septembre,929048.0,2025-09-30,OWONA Andela,PRUDENTIAL BENEFICIAL,8400.0,100.0,0.0,8400.0,Non,0.0,8400.0,2025
9,ASS-2025-10-000001,Octobre,932045.0,2025-10-06,AWOUNKENG Jeutang linda,PRUDENTIAL BENEFICIAL,14850.0,100.0,0.0,14850.0,Non,0.0,14850.0,2025
10,ASS-2025-10-000002,Octobre,932781.0,2025-10-08,AWOUNKENG Jeutang linda,PRUDENTIAL BENEFICIAL,6100.0,100.0,0.0,6100.0,Non,0.0,6100.0,2025


Nombre total de factures après nettoyage: 65


In [22]:
print("\n--- Étape 2: Classification des factures et calcul des écarts ---")

def classifier_factures(df):
    """Classifie chaque facture en fonction de son statut de paiement et des écarts.
    Les catégories sont: Payé Conforme, Payé avec Excédent, Payé avec Sous-paiement, Non Payé.
    """
    df_classified = df.copy()

    # Calculate the difference between 'montant_assurance' and 'montant_paye'
    df_classified['ecart'] = df_classified['montant_assurance'] - df_classified['montant_paye']

    # Define classification logic
    def classify_row(row):
        # A payment is considered 'compliant' if the difference is very small (due to potential floating point inaccuracies)
        # Or if the decompte_vu is 'Oui' and montant_assurance == montant_paye

        # First, check for 'decompte_vu' status
        if row['decompte_vu'].lower() != 'oui': # Assuming 'Non' or 'Non renseigné' means unpaid or not fully processed
            return 'Non Payé'

        # Now, if 'decompte_vu' is 'Oui', analyze the payment amounts
        if pd.isna(row['montant_paye']) or row['montant_paye'] == 0: # Explicitly zero or NaN implies not paid
             return 'Non Payé'

        # Use a small tolerance for floating point comparisons
        tolerance = 0.01
        if abs(row['ecart']) < tolerance: # Montant payé is very close to montant_assurance
            return 'Payé Conforme'
        elif row['ecart'] < -tolerance: # montant_assurance < montant_paye, meaning overpayment
            return 'Payé avec Excédent'
        elif row['ecart'] > tolerance: # montant_assurance > montant_paye, meaning underpayment
            return 'Payé avec Sous-paiement'
        else: # Fallback, should ideally be covered by 'Payé Conforme' or 'Non Payé'
            return 'Non Payé' # Default to non-paid if classification is ambiguous

    df_classified['statut_paiement'] = df_classified.apply(classify_row, axis=1)

    print("Classification des factures terminée.")
    return df_classified

def calculer_ecarts(df):
    """Calcule et ajoute les colonnes d'écarts pertinents au DataFrame.
    - ecart_brut: Montant Brut - Montant Payé
    - ecart_assurance: Montant Assurance - Montant Payé
    """
    df_ecarts = df.copy()
    df_ecarts['ecart_brut'] = df_ecarts['montant_brut'] - df_ecarts['montant_paye']
    # Note: 'ecart' was already calculated in classifier_factures for classification purpose
    # df_ecarts['ecart_assurance'] = df_ecarts['montant_assurance'] - df_ecarts['montant_paye']
    return df_ecarts

def calculer_synthese(df):
    """Calcule une synthèse des totaux par statut de paiement et autres métriques.
    """
    print("Calcul de la synthèse en cours...")

    total_factures = len(df)

    # Group by statut_paiement for summary
    synthese_status = df.groupby('statut_paiement').agg(
        total_montant_assurance=('montant_assurance', 'sum'),
        total_montant_paye=('montant_paye', 'sum'),
        nombre_factures=('id', 'count')
    ).reset_index()

    # Add a row for overall totals
    total_row = pd.DataFrame({
        'statut_paiement': ['Total Général'],
        'total_montant_assurance': [synthese_status['total_montant_assurance'].sum()],
        'total_montant_paye': [synthese_status['total_montant_paye'].sum()],
        'nombre_factures': [synthese_status['nombre_factures'].sum()]
    })
    synthese_status = pd.concat([synthese_status, total_row], ignore_index=True)

    print("Synthèse des calculs terminée.")
    return synthese_status

# --- Application des fonctions ---

df_processed = classifier_factures(df_combined)
df_processed = calculer_ecarts(df_processed)
synthese_df = calculer_synthese(df_processed)

print("\nAperçu des données après classification et calcul des écarts:")
display(df_processed.head())

print("\nSynthèse des statuts de paiement:")
display(synthese_df)


--- Étape 2: Classification des factures et calcul des écarts ---
Classification des factures terminée.
Calcul de la synthèse en cours...
Synthèse des calculs terminée.

Aperçu des données après classification et calcul des écarts:


,id,mois,facture,date_facture,patient,assurance,montant_brut,ticket,moderateur,montant_assurance,decompte_vu,montant_paye,solde,annee,ecart,statut_paiement,ecart_brut
0,ASS-2025-08-000001,Aout,912559.0,2025-08-22,TAKAM Magoum emelie josiane,PRUDENTIAL BENEFICIAL,15300.0,80.0,3060.0,12240.0,Non,0.0,12240.0,2025,12240.0,Non Payé,15300.0
4,ASS-2025-09-000001,Septembre,929019.0,2025-09-30,OWONA Andela,PRUDENTIAL BENEFICIAL,228400.0,100.0,0.0,228400.0,Non,0.0,228400.0,2025,228400.0,Non Payé,228400.0
5,ASS-2025-09-000002,Septembre,929048.0,2025-09-30,OWONA Andela,PRUDENTIAL BENEFICIAL,8400.0,100.0,0.0,8400.0,Non,0.0,8400.0,2025,8400.0,Non Payé,8400.0
9,ASS-2025-10-000001,Octobre,932045.0,2025-10-06,AWOUNKENG Jeutang linda,PRUDENTIAL BENEFICIAL,14850.0,100.0,0.0,14850.0,Non,0.0,14850.0,2025,14850.0,Non Payé,14850.0
10,ASS-2025-10-000002,Octobre,932781.0,2025-10-08,AWOUNKENG Jeutang linda,PRUDENTIAL BENEFICIAL,6100.0,100.0,0.0,6100.0,Non,0.0,6100.0,2025,6100.0,Non Payé,6100.0



Synthèse des statuts de paiement:


,statut_paiement,total_montant_assurance,total_montant_paye,nombre_factures
0,Non Payé,1220292.5,0.0,64
1,Payé Conforme,42720.0,42720.0,1
2,Total Général,1263012.5,42720.0,65


In [23]:
print("\n--- Étape 3: Contrôles de cohérence des données ---")

def controler_calculs(df_processed, synthese_df):
    """Effectue des contrôles rigoureux pour s'assurer de l'intégrité et de la cohérence des données.
    Retourne un dictionnaire de résultats de contrôle.
    """
    controles_results = {}

    # Contrôle 1: Cohérence de la somme des montants d'assurance entre le DataFrame détaillé et la synthèse
    total_assurance_df = df_processed['montant_assurance'].sum()
    total_assurance_synthese = synthese_df[synthese_df['statut_paiement'] == 'Total Général']['total_montant_assurance'].iloc[0]
    controles_results['Somme Montant Assurance Cohérente'] = abs(total_assurance_df - total_assurance_synthese) < 0.01
    if not controles_results['Somme Montant Assurance Cohérente']:
        print(f"ALERTE: Incohérence des totaux 'Montant Assurance'. Détail: {total_assurance_df}, Synthèse: {total_assurance_synthese}")

    # Contrôle 2: Cohérence de la somme des montants payés entre le DataFrame détaillé et la synthèse
    total_paye_df = df_processed['montant_paye'].sum()
    total_paye_synthese = synthese_df[synthese_df['statut_paiement'] == 'Total Général']['total_montant_paye'].iloc[0]
    controles_results['Somme Montant Payé Cohérente'] = abs(total_paye_df - total_paye_synthese) < 0.01
    if not controles_results['Somme Montant Payé Cohérente']:
        print(f"ALERTE: Incohérence des totaux 'Montant Payé'. Détail: {total_paye_df}, Synthèse: {total_paye_synthese}")

    # Contrôle 3: Vérification que 'ecart' = 'montant_assurance' - 'montant_paye' pour toutes les lignes
    df_processed['ecart_verif'] = df_processed['montant_assurance'] - df_processed['montant_paye']
    ecart_check = (abs(df_processed['ecart'] - df_processed['ecart_verif']) < 0.01).all()
    controles_results['Calcul Ecart Cohérent'] = ecart_check
    if not ecart_check:
        print("ALERTE: Des incohérences ont été détectées dans le calcul de la colonne 'ecart'.")
    df_processed.drop(columns=['ecart_verif'], inplace=True)

    # Contrôle 4: Vérification des ID uniques (si applicable)
    controles_results['ID Uniques'] = df_processed['id'].is_unique
    if not controles_results['ID Uniques']:
        print("ALERTE: Des identifiants de facture (ID) dupliqués ont été trouvés.")

    # Contrôle 5: Pas de montants négatifs pour montant_brut, montant_assurance, montant_paye
    cols_non_negatives = ['montant_brut', 'montant_assurance', 'montant_paye']
    for col in cols_non_negatives:
        if col in df_processed.columns:
            neg_check = (df_processed[col] >= 0).all()
            controles_results[f'{col} Non Négatif'] = neg_check
            if not neg_check:
                print(f"ALERTE: Des valeurs négatives ont été trouvées dans la colonne '{col}'.")

    # Contrôle 6: Vérifier que toutes les factures sont classifiées
    controles_results['Toutes Factures Classifiées'] = df_processed['statut_paiement'].notna().all()
    if not controles_results['Toutes Factures Classifiées']:
        print("ALERTE: Certaines factures n'ont pas été classifiées.")

    print("Contrôles de cohérence terminés.")
    return controles_results

# Appliquer les contrôles
controles = controler_calculs(df_processed, synthese_df)

print("\nRésultats des contrôles:")
for control, result in controles.items():
    status = "OK" if result else "ERREUR"
    print(f"- {control}: {status}")

# Vérifier si tous les contrôles sont OK avant de continuer
if all(controles.values()):
    print("\nTous les contrôles de cohérence ont été passés avec succès. Les données sont prêtes pour la génération du rapport.")
else:
    print("\nATTENTION: Certains contrôles de cohérence ont échoué. Veuillez examiner les alertes ci-dessus avant de procéder à la génération du rapport.")


--- Étape 3: Contrôles de cohérence des données ---
Contrôles de cohérence terminés.

Résultats des contrôles:
- Somme Montant Assurance Cohérente: OK
- Somme Montant Payé Cohérente: OK
- Calcul Ecart Cohérent: OK
- ID Uniques: OK
- montant_brut Non Négatif: OK
- montant_assurance Non Négatif: OK
- montant_paye Non Négatif: OK
- Toutes Factures Classifiées: OK

Tous les contrôles de cohérence ont été passés avec succès. Les données sont prêtes pour la génération du rapport.


In [24]:
print("\n--- Étape 4: Génération du Rapport Excel ---")

import pandas as pd
import openpyxl
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, Border, Side, Alignment, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles.differential import DifferentialStyle
from openpyxl.formatting.rule import Rule

def style_header(cell):
    cell.font = Font(bold=True, color="FFFFFF") # White text
    cell.fill = PatternFill(start_color="4F81BD", end_color="4F81BD", fill_type="solid") # Blue background
    cell.border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

def style_data_row(cell, is_total=False):
    cell.border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))
    if is_total:
        cell.font = Font(bold=True)
        cell.fill = PatternFill(start_color="DCE6F1", end_color="DCE6F1", fill_type="solid") # Light blue for totals

def set_column_widths(sheet, df):
    for i, col in enumerate(df.columns):
        max_length = 0
        # Consider header length
        max_length = max(max_length, len(str(col)))
        # Consider data length (up to a certain number of rows for performance)
        for j in range(min(len(df), 100)):
            max_length = max(max_length, len(str(df.iloc[j, i])))
        adjusted_width = (max_length + 2) * 1.2 # Add some padding
        sheet.column_dimensions[get_column_letter(i + 1)].width = adjusted_width if adjusted_width > 10 else 10

def creer_feuille_synthese(writer, df_processed, synthese_df, insurance_name):
    """Crée la feuille 'Synthèse' dans le rapport Excel avec des détails par année."""
    sheet_name = 'Synthèse'
    workbook = writer.book
    sheet = workbook.create_sheet(sheet_name)

    start_row = 1

    # --- Titre général ---
    sheet.cell(row=start_row, column=1, value=f"Synthèse du rapprochement financier pour {insurance_name}")
    sheet.cell(row=start_row, column=1).font = Font(size=16, bold=True)
    start_row += 2

    # --- Section: Synthèse par Année (2025) ---
    df_2025_summary = df_processed[df_processed['annee'] == 2025].groupby('statut_paiement').agg(
        total_montant_assurance=('montant_assurance', 'sum'),
        total_montant_paye=('montant_paye', 'sum'),
        nombre_factures=('id', 'count')
    ).reset_index()
    total_row_2025 = pd.DataFrame([{
        'statut_paiement': 'Total Général 2025',
        'total_montant_assurance': df_2025_summary['total_montant_assurance'].sum(),
        'total_montant_paye': df_2025_summary['total_montant_paye'].sum(),
        'nombre_factures': df_2025_summary['nombre_factures'].sum()
    }])
    df_2025_summary = pd.concat([df_2025_summary, total_row_2025], ignore_index=True)

    sheet.cell(row=start_row, column=1, value="Synthèse 2025")
    sheet.cell(row=start_row, column=1).font = Font(size=12, bold=True)
    start_row += 1
    # Write header for 2025 summary
    for col_idx, col_name in enumerate(df_2025_summary.columns):
        cell = sheet.cell(row=start_row, column=col_idx + 1, value=col_name.replace('_', ' ').capitalize())
        style_header(cell)
    start_row += 1
    # Write data for 2025 summary
    for r_idx, row_data in df_2025_summary.iterrows():
        is_total_row = "Total Général" in row_data['statut_paiement']
        for c_idx, value in enumerate(row_data):
            cell = sheet.cell(row=start_row + r_idx, column=c_idx + 1, value=value)
            style_data_row(cell, is_total=is_total_row)
            if c_idx in [1, 2]: # Monetary columns
                cell.number_format = '#,##0 "FCFA"'
            elif c_idx == 3: # Number of invoices
                cell.number_format = '#,##0'
    start_row += len(df_2025_summary) + 2

    # --- Section: Synthèse par Année (2026) ---
    df_2026_summary = df_processed[df_processed['annee'] == 2026].groupby('statut_paiement').agg(
        total_montant_assurance=('montant_assurance', 'sum'),
        total_montant_paye=('montant_paye', 'sum'),
        nombre_factures=('id', 'count')
    ).reset_index()
    total_row_2026 = pd.DataFrame([{
        'statut_paiement': 'Total Général 2026',
        'total_montant_assurance': df_2026_summary['total_montant_assurance'].sum(),
        'total_montant_paye': df_2026_summary['total_montant_paye'].sum(),
        'nombre_factures': df_2026_summary['nombre_factures'].sum()
    }])
    df_2026_summary = pd.concat([df_2026_summary, total_row_2026], ignore_index=True)

    sheet.cell(row=start_row, column=1, value="Synthèse 2026")
    sheet.cell(row=start_row, column=1).font = Font(size=12, bold=True)
    start_row += 1
    # Write header for 2026 summary
    for col_idx, col_name in enumerate(df_2026_summary.columns):
        cell = sheet.cell(row=start_row, column=col_idx + 1, value=col_name.replace('_', ' ').capitalize())
        style_header(cell)
    start_row += 1
    # Write data for 2026 summary
    for r_idx, row_data in df_2026_summary.iterrows():
        is_total_row = "Total Général" in row_data['statut_paiement']
        for c_idx, value in enumerate(row_data):
            cell = sheet.cell(row=start_row + r_idx, column=c_idx + 1, value=value)
            style_data_row(cell, is_total=is_total_row)
            if c_idx in [1, 2]: # Monetary columns
                cell.number_format = '#,##0 "FCFA"'
            elif c_idx == 3: # Number of invoices
                cell.number_format = '#,##0'
    start_row += len(df_2026_summary) + 2

    # --- Section: Synthèse Générale (Toutes Années) ---
    sheet.cell(row=start_row, column=1, value="Synthèse Générale")
    sheet.cell(row=start_row, column=1).font = Font(size=12, bold=True)
    start_row += 1
    # Write header for general summary
    for col_idx, col_name in enumerate(synthese_df.columns):
        cell = sheet.cell(row=start_row, column=col_idx + 1, value=col_name.replace('_', ' ').capitalize())
        style_header(cell)
    start_row += 1
    # Write data for general summary
    for r_idx, row_data in synthese_df.iterrows():
        is_total_row = "Total Général" in row_data['statut_paiement']
        for c_idx, value in enumerate(row_data):
            cell = sheet.cell(row=start_row + r_idx, column=c_idx + 1, value=value)
            style_data_row(cell, is_total=is_total_row)
            if c_idx in [1, 2]: # Monetary columns
                cell.number_format = '#,##0 "FCFA"'
            elif c_idx == 3: # Number of invoices
                cell.number_format = '#,##0'
    start_row += len(synthese_df) + 2

    # --- Section: Répartition des Factures (Overall) ---
    sheet.cell(row=start_row, column=1, value="Répartition des Factures (Toutes Années)")
    sheet.cell(row=start_row, column=1).font = Font(size=12, bold=True)
    start_row += 1

    repartition_df = df_processed['statut_paiement'].value_counts(normalize=True).reset_index()
    repartition_df.columns = ['Statut de Paiement', 'Pourcentage']
    repartition_df['Nombre de Factures'] = df_processed['statut_paiement'].value_counts().values
    repartition_df['Pourcentage'] = repartition_df['Pourcentage'].map(lambda x: f"{x:.2%}")

    # Write header for repartition
    for col_idx, col_name in enumerate(repartition_df.columns):
        cell = sheet.cell(row=start_row, column=col_idx + 1, value=col_name)
        style_header(cell)
    start_row += 1
    # Write data for repartition
    for r_idx, row_data in repartition_df.iterrows():
        for c_idx, value in enumerate(row_data):
            cell = sheet.cell(row=start_row + r_idx, column=c_idx + 1, value=value)
            style_data_row(cell)
            if c_idx == 2: # Number of invoices
                cell.number_format = '#,##0'
    start_row += len(repartition_df) + 2

    # --- Section: Principaux Points à Réclamer ---
    sheet.cell(row=start_row, column=1, value="Principaux Points à Réclamer")
    sheet.cell(row=start_row, column=1).font = Font(size=12, bold=True)
    start_row += 1

    # Calculate total debt
    total_debt = df_processed[df_processed['statut_paiement'].isin(['Non Payé', 'Payé avec Sous-paiement'])]['ecart'].sum()
    sheet.cell(row=start_row, column=1, value=f"Dette actuelle de l'assurance : {total_debt:,.0f} FCFA").font = Font(bold=True, color="FF0000") # Red for debt
    start_row += 2

    # Filter for non-compliant invoices
    df_claims = df_processed[df_processed['statut_paiement'] != 'Payé Conforme'].copy()
    if not df_claims.empty:
        # Group by patient and sum the écart for top claimants
        top_claimants = df_claims.groupby(['patient', 'statut_paiement'])['ecart'].sum().abs().nlargest(5).reset_index()
        top_claimants.columns = ['Patient', 'Statut de Paiement', 'Montant à Réclamer (Absolu)']

        # Write header for top claimants
        for col_idx, col_name in enumerate(top_claimants.columns):
            cell = sheet.cell(row=start_row, column=col_idx + 1, value=col_name)
            style_header(cell)
        start_row += 1
        # Write data for top claimants
        for r_idx, row_data in top_claimants.iterrows():
            for c_idx, value in enumerate(row_data):
                cell = sheet.cell(row=start_row + r_idx, column=c_idx + 1, value=value)
                style_data_row(cell)
                if c_idx == 2: # Monetary column
                    cell.number_format = '#,##0 "FCFA"'
    else:
        sheet.cell(row=start_row, column=1, value="Aucun point majeur à réclamer.").font = Font(italic=True)

    start_row += len(top_claimants) + 2 if not df_claims.empty else 2

    sheet.cell(row=start_row, column=1, value="Actions Stratégiques Recommandées").font = Font(size=12, bold=True)
    start_row += 1
    sheet.cell(row=start_row, column=1, value="1. Mettre en place un système de suivi des paiements en temps réel.").font = Font(italic=True)
    start_row += 1
    sheet.cell(row=start_row, column=1, value="2. Négocier des accords de paiement avec les assureurs en cas de sous-paiement chronique.").font = Font(italic=True)
    start_row += 1
    sheet.cell(row=start_row, column=1, value="3. Revoir les processus internes de facturation pour minimiser les erreurs.").font = Font(italic=True)
    start_row += 1
    sheet.cell(row=start_row, column=1, value="4. Organiser des sessions de formation pour le personnel sur les procédures de rapprochement.").font = Font(italic=True)
    start_row += 1

    # Adjust column widths for the entire sheet
    set_column_widths(sheet, df_processed.iloc[:, :len(synthese_df.columns)]) # Use synthese_df columns as a base for width for clarity
    print(f"Feuille '{sheet_name}' créée.")

def creer_feuille_reclamations(writer, df_processed, insurance_name):
    """Crée la feuille 'Réclamations' avec les factures à traiter (non conformes) et formatage."""
    sheet_name = 'Réclamations'
    workbook = writer.book
    sheet = workbook.create_sheet(sheet_name)

    # Filtrer les factures qui ne sont pas 'Payé Conforme'
    df_reclamations = df_processed[df_processed['statut_paiement'] != 'Payé Conforme'].copy()

    if df_reclamations.empty:
        title_message = "Aucune réclamation à traiter (toutes les factures sont conformes)."
    else:
        title_message = f"Liste des réclamations pour {insurance_name}"

    # Titre
    sheet.cell(row=1, column=1, value=title_message)
    sheet.cell(row=1, column=1).font = Font(size=14, bold=True)

    if not df_reclamations.empty:
        # Write header
        for col_idx, col_name in enumerate(df_reclamations.columns):
            cell = sheet.cell(row=3, column=col_idx + 1, value=col_name.replace('_', ' ').capitalize())
            style_header(cell)

        # Write data
        for r_idx, row_data in df_reclamations.iterrows():
            for c_idx, value in enumerate(row_data):
                cell = sheet.cell(row=4 + r_idx, column=c_idx + 1, value=value)
                style_data_row(cell)
                col_name = df_reclamations.columns[c_idx]
                if col_name in ['montant_brut', 'montant_assurance', 'montant_paye', 'ticket', 'moderateur', 'solde', 'ecart', 'ecart_brut']:
                    cell.number_format = '#,##0 "FCFA"'
                elif col_name == 'date_facture':
                    cell.number_format = 'DD/MM/YYYY'
                cell.alignment = Alignment(horizontal='center' if col_name not in ['patient', 'id'] else 'left')

        # Conditional formatting for 'ecart' column
        ecart_col_idx = df_reclamations.columns.get_loc('ecart') + 1

        # Red for positive écart (underpayment)
        red_fill_dxf = DifferentialStyle(fill=PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid'), font=Font(color='9C0006'))
        sheet.conditional_formatting.add(
            f'{get_column_letter(ecart_col_idx)}4:{get_column_letter(ecart_col_idx)}{3 + len(df_reclamations)}',
            Rule(type='cellIs', operator='greaterThan', formula=['0'], dxf=red_fill_dxf)
        )
        # Green for negative écart (overpayment)
        green_fill_dxf = DifferentialStyle(fill=PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid'), font=Font(color='006100'))
        sheet.conditional_formatting.add(
            f'{get_column_letter(ecart_col_idx)}4:{get_column_letter(ecart_col_idx)}{3 + len(df_reclamations)}',
            Rule(type='cellIs', operator='lessThan', formula=['0'], dxf=green_fill_dxf)
        )

    set_column_widths(sheet, df_reclamations)
    print(f"Feuille '{sheet_name}' créée.")

def creer_feuille_donnees_completes(writer, df_processed, insurance_name):
    """Crée la feuille 'Données Complètes' avec toutes les données traitées et formatage."""
    sheet_name = 'Données complètes'
    workbook = writer.book
    sheet = workbook.create_sheet(sheet_name)

    # Titre
    sheet.cell(row=1, column=1, value=f"Données complètes du rapprochement financier pour {insurance_name}")
    sheet.cell(row=1, column=1).font = Font(size=14, bold=True)

    # Write header
    for col_idx, col_name in enumerate(df_processed.columns):
        cell = sheet.cell(row=3, column=col_idx + 1, value=col_name.replace('_', ' ').capitalize())
        style_header(cell)

    # Write data
    for r_idx, row_data in df_processed.iterrows():
        for c_idx, value in enumerate(row_data):
            cell = sheet.cell(row=4 + r_idx, column=c_idx + 1, value=value)
            style_data_row(cell)
            col_name = df_processed.columns[c_idx]
            if col_name in ['montant_brut', 'montant_assurance', 'montant_paye', 'ticket', 'moderateur', 'solde', 'ecart', 'ecart_brut']:
                cell.number_format = '#,##0 "FCFA"'
            elif col_name == 'date_facture':
                cell.number_format = 'DD/MM/YYYY'
            cell.alignment = Alignment(horizontal='center' if col_name not in ['patient', 'id'] else 'left')

    # Conditional formatting for 'ecart' column
    if 'ecart' in df_processed.columns:
        ecart_col_idx = df_processed.columns.get_loc('ecart') + 1

        # Red for positive écart (underpayment)
        red_fill_dxf = DifferentialStyle(fill=PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid'), font=Font(color='9C0006'))
        sheet.conditional_formatting.add(
            f'{get_column_letter(ecart_col_idx)}4:{get_column_letter(ecart_col_idx)}{3 + len(df_processed)}',
            Rule(type='cellIs', operator='greaterThan', formula=['0'], dxf=red_fill_dxf)
        )
        # Green for negative écart (overpayment)
        green_fill_dxf = DifferentialStyle(fill=PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid'), font=Font(color='006100'))
        sheet.conditional_formatting.add(
            f'{get_column_letter(ecart_col_idx)}4:{get_column_letter(ecart_col_idx)}{3 + len(df_processed)}',
            Rule(type='cellIs', operator='lessThan', formula=['0'], dxf=green_fill_dxf)
        )

    set_column_widths(sheet, df_processed)
    print(f"Feuille '{sheet_name}' créée.")

def generer_rapport_excel(df_processed, synthese_df, insurance_name):
    """Génère le rapport Excel complet avec les trois feuilles."""
    output_filename = f"Rapport_Rapprochement_Assurance_{insurance_name}.xlsx"
    print(f"Génération du rapport Excel: {output_filename}")

    with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        # Ensure 'Synthèse' is the first sheet by creating it directly
        creer_feuille_synthese(writer, df_processed, synthese_df, insurance_name)
        # The next sheets are created by openpyxl's default behavior, or explicitly added
        creer_feuille_reclamations(writer, df_processed, insurance_name)
        creer_feuille_donnees_completes(writer, df_processed, insurance_name)

    print(f"Rapport Excel '{output_filename}' généré avec succès.")
    # Optionally, download the file if running in Colab
    try:
        from google.colab import files
        files.download(output_filename)
        print(f"Le fichier '{output_filename}' a été téléchargé.")
    except ImportError:
        print("Pour télécharger le fichier, exécutez ce code dans Google Colab.")

# --- Application de la fonction de génération de rapport ---
# Ensure the 'annee' column is available in df_processed for the new synthesis logic.
# This should already be done in previous steps, but it's crucial for the new creer_feuille_synthese.

if all(controles.values()): # Only generate report if all controls passed
    generer_rapport_excel(df_processed, synthese_df, INSURANCE_NAME)
else:
    print("La génération du rapport est annulée en raison d'échecs de contrôle de cohérence.")


--- Étape 4: Génération du Rapport Excel ---
Génération du rapport Excel: Rapport_Rapprochement_Assurance_Ficial.xlsx
Feuille 'Synthèse' créée.
Feuille 'Réclamations' créée.
Feuille 'Données complètes' créée.
Rapport Excel 'Rapport_Rapprochement_Assurance_Ficial.xlsx' généré avec succès.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Le fichier 'Rapport_Rapprochement_Assurance_Ficial.xlsx' a été téléchargé.
